# MOZYME en GPU con criterio estricto: depuración, ronda 5

Ronda 4 (commit 91eb7992, `SCFCRT=1.D-6 THRESH=1.D-15`):

- **Ubiquitina:** resuelto. 51 iteraciones en la GPU (10 s), calor -5517.92812 (CPU -5517.92809), norma del
  gradiente 1574.43646 (CPU 1574.43648). Crambina y ADN también convergen.
- **Barnasa:** la etapa de densidad ya funciona y el SCF en la GPU llega a la convergencia, pero en la última
  iteración un LMO virtual pasa de 2048 coeficientes, el límite de la tabla en memoria compartida de `diagg1`,
  y el cálculo vuelve a la CPU (477 s). Además se agotaron las 256 tablas en memoria global.
  Corregido: un LMO que solo ya no cabe va directo a memoria global; las rotaciones aceptan hasta 9216
  coeficientes por LMO; hay tantas tablas globales como LMO virtuales (si hay memoria).

Esta ronda verifica que el commit compilado sea el último de la rama (si no, se detiene).

**Uso:** Entorno de ejecución → **Desconectar y borrar el entorno de ejecución** → GPU A100 → Ejecutar todas.
~15 min de compilación + ~5 min de corridas. Al final se descarga `mozyme_strict_debug.zip`: déjelo en Descargas.

In [ ]:
BRANCH = 'paper-bench'
import os, re, subprocess, sys, time, json, shutil
from pathlib import Path

def run(cmd, check=True):
    print('$', cmd, flush=True)
    p = subprocess.run(cmd, shell=True, executable='/bin/bash', capture_output=True, text=True)
    out = (p.stdout + p.stderr)[-3000:]
    if out.strip(): print(out)
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv,noheader')
CC = subprocess.run('nvidia-smi --query-gpu=compute_cap --format=csv,noheader', shell=True, capture_output=True,
                    text=True).stdout.strip().split('\n')[0].replace('.', '')
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libopenblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
if SRC.exists():   # a reused runtime: bring the clone to the current commit (round 2 built the old one)
    run(f'git -C {SRC} fetch -q --depth 1 origin {BRANCH} && git -C {SRC} reset -q --hard FETCH_HEAD')
else:
    run(f'git clone -q --depth 1 -b {BRANCH} https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
remote = subprocess.run(f'git ls-remote https://github.com/juvenalyosa/mopac_gpu.git refs/heads/{BRANCH}',
                        shell=True, capture_output=True, text=True).stdout.split()[0]
head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
print('HEAD', head, ' remoto', remote)
if head != remote:
    raise SystemExit('El código clonado no es el último commit de la rama: desconecte y borre el entorno y repita.')
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || tail -30 /content/cmake.log')
t0 = time.time()
rc = run(f'cmake --build {BUILD} --target mopac --parallel {os.cpu_count()} > /content/build.log 2>&1', check=False)
if rc: run('grep -i -B2 -A8 error /content/build.log | head -60', check=False); raise SystemExit('build failed')
print(f'compilado en {(time.time()-t0)/60:.1f} min')
MOPAC = BUILD / 'mopac'

In [ ]:
PUB = SRC / 'benchmarks/publication_inputs/mop'
WORK = Path('/content/mozyme_strict_debug'); WORK.mkdir(exist_ok=True)
STRICT = 'SCFCRT=1.D-6 THRESH=1.D-15'
BASE = 'PM7 GEO_DAT="{pdb}" 1SCF MOZYME MOZYME_GPU MOZYME_MINBLK=16 PULAY SHIFT=-50 ITRY={itry} GEO-OK NOCOMMENTS GRADIENTS AUX {crit}'
PDB = {'ubiquitin': 'protein_ubiquitin_1ubq_hydrogenated.pdb', 'barnase': 'protein_barnase_1rnb_hydrogenated.pdb',
       'crambin': 'protein_crambin_1crn_hydrogenated.pdb', 'dna': 'dna_dodecamer_1bna_hydrogenated.pdb'}
TRACE = {'MOPAC_MOZYME_SCF_TRACE': '1', 'MOPAC_GPU_DEBUG': '1'}
CPU = {'MOPAC_NOGPU': '1', 'OMP_NUM_THREADS': '1'}
NORES = {'MOPAC_MOZYME_RESIDENT_SCF': '0', 'MOPAC_MOZYME_SCF_GPU': '0'}   # SCF loop on the host, rest on the GPU
# name, protein, ITRY, criterion, extra keywords, environment
CASES = [
    ('ubq_gpu', 'ubiquitin', 200,  STRICT,         '',    TRACE),
    ('bar_gpu', 'barnase',   200,  STRICT,         '',    TRACE),
    ('cra_gpu', 'crambin',   200,  STRICT,         '',    TRACE),
    ('dna_gpu', 'dna',       200,  STRICT,         '',    TRACE),
]
# CPU references, same decks (rounds 1-3, Colab Xeon; barnase and ubiquitin: SCF loop on the host)
REF = {'ubq_gpu': (-5517.92809, 1574.43648), 'bar_gpu': (-7232.37991, 2051.01374)}

def summarize(name, d, wall):
    out = (d / 'job.out').read_text(errors='ignore') if (d / 'job.out').exists() else ''
    so = (d / 'job.stdout').read_text(errors='ignore') if (d / 'job.stdout').exists() else ''
    txt = out + '\n' + so
    g = lambda pat: (re.findall(pat, txt) or [None])[-1]
    traces = re.findall(r'\[MOZYME GPU SCF trace\] iter=(\d+) escf=(\S+) de=(\S+) ovmax=(\S+)', txt)
    cpu_its = re.findall(r'^ ITER\.\s+(\d+) PLS=\s*(\S+)', txt, re.M)
    return {'case': name, 'wall_s': round(wall, 1), 'heat': g(r'FINAL HEAT OF FORMATION =\s+(\S+)'),
            'gnorm': g(r'GRADIENT NORM\s+=\s+(\S+)'),
            'scf_failed': bool(re.search(r'SCF (FAILED|NOT ACHIEVED)|UNABLE TO ACHIEVE', txt)),
            'gpu_iterations': g(r'\[MOZYME GPU SCF\] device_id=.*iterations=\s*(\d+)'),
            'fallback_cpu': len(re.findall(r'status=fallback_cpu reason', txt)),
            'storage_grown': len(re.findall(r'lmo_storage_grown', txt)),
            'last_trace': traces[-1] if traces else None, 'cpu_last_iter': cpu_its[-1] if cpu_its else None,
            'shell_spill_max': max([int(x) for x in re.findall(r'shell_spill=(\d+)', txt)] or [None]) if 'shell_spill=' in txt else None,
            'shell_overflow_max': max([int(x) for x in re.findall(r'shell_overflow=(\d+)', txt)] or [None]) if 'shell_overflow=' in txt else None,
            'ref_cpu': REF.get(name),
            'debug': re.findall(r'\[MOZYME GPU debug\].*', txt)[:6]}

results = []
for name, prot, itry, crit, extra, env in CASES:
    d = WORK / name; d.mkdir(exist_ok=True)
    shutil.copy2(PUB / PDB[prot], d / PDB[prot])
    (d / 'job.mop').write_text(BASE.format(pdb=PDB[prot], itry=itry, crit=crit) + extra + '\n' + prot + '\n\n')
    full = dict(os.environ); full.update(env)
    print(f'== {name} ...', flush=True)
    t0 = time.time()
    with open(d / 'job.stdout', 'w') as fh:
        try:
            subprocess.run([str(MOPAC), 'job.mop'], cwd=d, env=full, stdout=fh, stderr=subprocess.STDOUT,
                           timeout=1800)
        except subprocess.TimeoutExpired:
            print('   timeout (30 min)')
    r = summarize(name, d, time.time() - t0); results.append(r)
    print(json.dumps(r, indent=1), flush=True)
(WORK / 'summary.json').write_text(json.dumps(results, indent=1))

In [ ]:
run(f'nvidia-smi > {WORK}/nvidia-smi.txt; git -C {SRC} log -1 > {WORK}/commit.txt', check=False)
run(f'cd /content && rm -f mozyme_strict_debug.zip && zip -q -r mozyme_strict_debug.zip mozyme_strict_debug '
    f'-x "*.pdb" "*.aux"')
from google.colab import files
files.download('/content/mozyme_strict_debug.zip')